# 02 — Feature Engineering

Objective: build defensible features with zero target leakage. Each feature states what it is, why it may help, and why it is safe. Fitted state (ticket counts) is learned on training folds only.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils import set_global_seed
from configs.config import RANDOM_STATE
set_global_seed(RANDOM_STATE)
%matplotlib inline

In [ ]:
from src.data_loading import load_raw
from src.feature_engineering import TitanicFeatureEngineer, add_features_with_combined_ticket_counts, extract_title
train_df, test_df = load_raw(ROOT/'data'/'raw'/'train.csv', ROOT/'data'/'raw'/'test.csv')
print(train_df['Name'].head(3).tolist())
print(train_df['Name'].apply(extract_title).value_counts())

## Title / Surname
`Title` groups rare titles to `Rare` (Mlle→Miss, Mme→Mrs). Measured survival: Mrs 79.37%, Miss 70.27%, Master 57.50%, Rare 34.78%, Mr 15.67%. `Surname` is extracted for EDA only — high cardinality, never modelled.


## Family: size, alone flag, categories
`FamilySize = SibSp + Parch + 1`; `IsAlone`; Alone/Small(2–4)/Large(5+). Measured: Small 57.88%, Alone 30.35%, Large 16.13% survival — a clear non-linearity raw counts hide.


## Ticket groups and per-person fare
`TicketGroupSize` counts sharers (fit on train only in the pipeline; combined counts used here purely for EDA plots). `FarePerPerson = Fare / TicketGroupSize` corrects per-ticket pricing. Safe division via clip(lower=1).


## Cabin deck + known flag; Mother; AgeGroup
`CabinDeck` (first letter, else Missing) + `CabinKnown`. Known cabins survive at 66.67% vs 29.99% unknown — confounded with class, so modelled as plain categories, not causal claims. `Mother` = adult female, Parch>0, Mrs/Miss — an approximate heuristic flag. `AgeGroup` (Child/Teen/Adult/Senior) is EDA-only; the model uses raw `Age`.


In [ ]:
tr_eng, te_eng = add_features_with_combined_ticket_counts(train_df, test_df)
print(tr_eng[['Title','FamilySize','IsAlone','FamilySizeCategory','TicketGroupSize','FarePerPerson','CabinDeck','CabinKnown','Mother','AgeGroup']].head())
print(tr_eng.groupby('Title')['Survived'].agg(['mean','count']))
print(tr_eng.groupby('FamilySizeCategory')['Survived'].agg(['mean','count']))
print(tr_eng.groupby('CabinKnown')['Survived'].agg(['mean','count']))
from src.visualization import plot_survival_by_category
plot_survival_by_category(tr_eng, 'FamilySizeCategory', ROOT/'reports'/'figures'/'07_survival_by_familysize.png', 'Survival rate by FamilySizeCategory')
plot_survival_by_category(tr_eng, 'Title', ROOT/'reports'/'figures'/'09_survival_by_title.png', 'Survival rate by Title')

## Leakage discipline
- Transformer never sees `Survived`. `fit` learns only ticket counts; `transform` is row-wise otherwise.
- In modelling (notebook 03/04) the engineer sits *inside* the Pipeline, so CV folds fit counts on train splits only.
- `PassengerId`, `Surname`, raw `Name/Ticket/Cabin`, and `AgeGroup` are excluded from the model matrix (`configs/config.py` → `DROP_COLUMNS`, `NUMERIC/CATEGORICAL_FEATURES`).


In [ ]:
eng = TitanicFeatureEngineer().fit(train_df)
print('ticket groups learned:', len(eng.ticket_counts_))
print(eng.transform(train_df.head(2)).filter(['Title','FamilySize','TicketGroupSize','FarePerPerson','CabinDeck','CabinKnown','Mother']).to_string())